**Import and Device set up**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.utils import resample
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import cycle
from PIL import Image
import torchvision.transforms.functional as TF
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

**Loss Function & Custom Modules (Attention & Fusion)**

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha 
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv1(x_cat))

class CBAM(nn.Module):
    def __init__(self, in_planes, ratio=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return x

class FeatureFusion(nn.Module):
    def __init__(self, c_low, c_high, out_channels=256):
        super().__init__()
        self.conv_low = nn.Conv2d(c_low, out_channels, 1)
        self.conv_high = nn.Conv2d(c_high, out_channels, 1)
        self.fuse = nn.Sequential(
            nn.Conv2d(out_channels * 2, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, f_low, f_high):
        f_high_up = F.interpolate(f_high, size=f_low.shape[2:], mode='bilinear', align_corners=False)
        f_low_proj = self.conv_low(f_low)
        f_high_proj = self.conv_high(f_high_up)
        combined = torch.cat([f_low_proj, f_high_proj], dim=1)
        return self.fuse(combined)

class SwappedQKVCrossAttention(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.q_eff = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.k_eff = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.v_eff = nn.Conv2d(in_channels, in_channels, kernel_size=1)

        self.q_r = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.k_r = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.v_r = nn.Conv2d(in_channels, in_channels, kernel_size=1)

        self.scale = in_channels ** -0.5 

    def forward(self, f_eff, f_r):
        B, C, H, W = f_eff.size()
        N = H * W 
        
        qe = self.q_eff(f_eff).view(B, C, N).permute(0, 2, 1) 
        ke = self.k_eff(f_eff).view(B, C, N)                  
        ve = self.v_eff(f_eff).view(B, C, N).permute(0, 2, 1) 

        qr = self.q_r(f_r).view(B, C, N).permute(0, 2, 1)     
        kr = self.k_r(f_r).view(B, C, N)                      
        vr = self.v_r(f_r).view(B, C, N).permute(0, 2, 1)     

        attn_top = torch.matmul(qr, ke) * self.scale
        attn_top = F.softmax(attn_top, dim=-1)
        out_top = torch.matmul(attn_top, ve) 

        attn_bot = torch.matmul(qe, kr) * self.scale
        attn_bot = F.softmax(attn_bot, dim=-1)
        out_bot = torch.matmul(attn_bot, vr) 

        out_top = out_top.permute(0, 2, 1).contiguous().view(B, C, H, W)
        out_bot = out_bot.permute(0, 2, 1).contiguous().view(B, C, H, W)

        return out_top + out_bot

class BidirectionalFeatureFusion(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.blend = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.blend(x)

**Main Model Architecture (DCAT)**

In [ ]:
class DCAT_Model_True(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        
        # --- ACETO BRANCH (EFFICIENTNET-B4) ---
        eff_aceto = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
        self.aceto_block3 = nn.Sequential(*eff_aceto.features[:4]) 
        self.aceto_block4 = nn.Sequential(*eff_aceto.features[4:6]) 
        
        # --- IODINE BRANCH (EFFICIENTNET-B4) ---
        eff_iodine = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
        self.iodine_block3 = nn.Sequential(*eff_iodine.features[:4]) 
        self.iodine_block4 = nn.Sequential(*eff_iodine.features[4:6]) 
        
        self.fuse_aceto = FeatureFusion(c_low=56, c_high=160, out_channels=256)
        self.fuse_iodine = FeatureFusion(c_low=56, c_high=160, out_channels=256)
        
        self.cross_attention = SwappedQKVCrossAttention(in_channels=256)
        self.bidirectional_fusion = BidirectionalFeatureFusion(channels=256)
        self.cbam = CBAM(in_planes=256)
        
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5), 
            nn.Linear(256, 128), 
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, aceto_img, iod_img):
        a_b3 = self.aceto_block3(aceto_img)
        a_b4 = self.aceto_block4(a_b3)
        feat_aceto = self.fuse_aceto(a_b3, a_b4)
        
        i_b3 = self.iodine_block3(iod_img)
        i_b4 = self.iodine_block4(i_b3)
        feat_iodine = self.fuse_iodine(i_b3, i_b4)
        
        attention_output = self.cross_attention(feat_aceto, feat_iodine)
        fused_output = self.bidirectional_fusion(attention_output)
        refined_output = self.cbam(fused_output)
        
        pooled = self.pool(refined_output).view(refined_output.size(0), -1)
        return self.classifier(pooled)

**Dataset Class & Data Preparation**

In [ ]:
class PairedMemoryDataset(Dataset):
    def __init__(self, aceto_list, iod_list, label_list, is_training=False):
        self.aceto_list = aceto_list
        self.iod_list = iod_list
        self.label_list = label_list
        self.is_training = is_training

    def __len__(self):
        return len(self.label_list)

    def __getitem__(self, idx):
        img_a_clean = self.aceto_list[idx]
        img_i_clean = self.iod_list[idx]
        label = self.label_list[idx]

        img_a_pil = Image.fromarray(img_a_clean)
        img_i_pil = Image.fromarray(img_i_clean)

        if self.is_training:
            if random.random() > 0.5:
                img_a_pil = TF.hflip(img_a_pil)
                img_i_pil = TF.hflip(img_i_pil)
            if random.random() > 0.5:
                img_a_pil = TF.vflip(img_a_pil)
                img_i_pil = TF.vflip(img_i_pil)

            angle = random.uniform(-20, 20)
            img_a_pil = TF.rotate(img_a_pil, angle)
            img_i_pil = TF.rotate(img_i_pil, angle)

        img_a_tensor = TF.to_tensor(TF.resize(img_a_pil, (224, 224)))
        img_i_tensor = TF.to_tensor(TF.resize(img_i_pil, (224, 224)))

        mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
        img_a_tensor = TF.normalize(img_a_tensor, mean=mean, std=std)
        img_i_tensor = TF.normalize(img_i_tensor, mean=mean, std=std)

        return img_a_tensor, img_i_tensor, torch.tensor(label, dtype=torch.long)


print("Splitting off 20% of patients for the fixed Test Set...")
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
cv_idx, test_idx = next(gss.split(images_aceto, Diagnosis, groups=patient_ids))

cv_a = [images_aceto[i] for i in cv_idx]
cv_i = [images_iod[i] for i in cv_idx]
cv_lbl = [Diagnosis[i] for i in cv_idx]
cv_pats = [patient_ids[i] for i in cv_idx]

test_a = [images_aceto[i] for i in test_idx]
test_i = [images_iod[i] for i in test_idx]
test_lbl = [Diagnosis[i] for i in test_idx]

test_dataset = PairedMemoryDataset(test_a, test_i, test_lbl, is_training=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

target_names = ['Normal', 'CIN1', 'High Grade']
fold_model_paths = []
n_splits = 5
num_epochs = 35
patience_limit = 7 

sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
fold_metrics = {'Accuracy': [], 'Precision': [], 'Recall': [], 'F1_Score': [], 'AUC': []}

**K-Fold Training Loop**

In [ ]:
for fold, (train_idx, val_idx) in enumerate(sgkf.split(cv_a, cv_lbl, groups=cv_pats)):
    print(f"\n{'='*50}")
    print(f"========== FOLD {fold + 1}/{n_splits} ==========")
    print(f"{'='*50}")
    
    train_a_fold = [cv_a[i] for i in train_idx]
    train_i_fold = [cv_i[i] for i in train_idx]
    train_lbl_fold = [cv_lbl[i] for i in train_idx]
    
    val_a_fold = [cv_a[i] for i in val_idx]
    val_i_fold = [cv_i[i] for i in val_idx]
    val_lbl_fold = [cv_lbl[i] for i in val_idx]
    
    train_dataset = PairedMemoryDataset(train_a_fold, train_i_fold, train_lbl_fold, is_training=True)
    val_dataset = PairedMemoryDataset(val_a_fold, val_i_fold, val_lbl_fold, is_training=False)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
    
    model = DCAT_Model_True(num_classes=3).to(device)
    
    alpha_weights = torch.tensor([0.8915, 0.8779, 1.3529], dtype=torch.float32).to(device)
    criterion = FocalLoss(gamma=2.0, alpha=alpha_weights) 
    
    # STEP 1: FREEZE THE EFFICIENTNET BACKBONES
    for param in model.aceto_block3.parameters(): param.requires_grad = False
    for param in model.aceto_block4.parameters(): param.requires_grad = False
    for param in model.iodine_block3.parameters(): param.requires_grad = False
    for param in model.iodine_block4.parameters(): param.requires_grad = False
    
    # STEP 2: OPTIMIZER ONLY TRAINS THE CUSTOM HEADS (With L2 Regularization)
    optimizer = torch.optim.Adam([
        {'params': model.fuse_aceto.parameters()},
        {'params': model.fuse_iodine.parameters()},
        {'params': model.cross_attention.parameters()},
        {'params': model.bidirectional_fusion.parameters()},
        {'params': model.cbam.parameters()},
        {'params': model.classifier.parameters()}
    ], lr=1e-4, weight_decay=1e-4)
    
    best_val_f1 = 0.0 
    epochs_without_improvement = 0
    fold_model_path = f'best_model_fold_{fold+1}.pth'
    fold_model_paths.append(fold_model_path)
    
    for epoch in range(num_epochs):
        
        # STEP 3: UNFREEZE EVERYTHING AT EPOCH 5
        if epoch == 5:
            print("\n🔓 Unfreezing EfficientNet Backbones for Fine-Tuning...")
            for param in model.parameters():
                param.requires_grad = True
                
            optimizer = torch.optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-4)
            
        model.train()
        running_loss = 0.0
        
        for batch_a, batch_i, labels in train_loader:
            batch_a, batch_i, labels = batch_a.to(device), batch_i.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(batch_a, batch_i)
            loss = criterion(outputs, labels) 
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
        model.eval()
        val_preds, val_trues = [], []
        
        with torch.no_grad():
            for batch_a, batch_i, labels in val_loader:
                batch_a, batch_i = batch_a.to(device), batch_i.to(device)
                outputs = model(batch_a, batch_i)
                _, predicted = torch.max(outputs.data, 1)
                val_preds.extend(predicted.cpu().numpy())
                val_trues.extend(labels.numpy())
        
        current_val_f1 = f1_score(val_trues, val_preds, average='macro')
        print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {running_loss/len(train_loader):.4f} | Val Macro-F1: {current_val_f1:.4f}")
        
        if current_val_f1 > best_val_f1:
            best_val_f1 = current_val_f1
            torch.save(model.state_dict(), fold_model_path)
            print(f"   -> 🌟 New best model saved! (F1: {best_val_f1:.4f})")
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            print(f"   -> No improvement for {epochs_without_improvement} epoch(s).")
            
        if epochs_without_improvement >= patience_limit:
            print(f"🛑 Early stopping triggered at epoch {epoch+1}. Moving to next fold.")
            break
            
    # Fold Evaluation on Holdout Test Set
    print(f"\nEvaluating Fold {fold + 1} Best Model on FIXED TEST SET...")
    model.load_state_dict(torch.load(fold_model_path)) 
    model.eval()
    
    fold_preds, fold_trues, fold_probs = [], [], []
    
    with torch.no_grad():
        for batch_a, batch_i, labels in test_loader:
            batch_a, batch_i = batch_a.to(device), batch_i.to(device)
            outputs = model(batch_a, batch_i)
            probs = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs.data, 1)
            
            fold_probs.extend(probs.cpu().numpy())
            fold_preds.extend(predicted.cpu().numpy())
            fold_trues.extend(labels.numpy())
            
    print(classification_report(fold_trues, fold_preds, target_names=target_names))
    
    fold_metrics['Accuracy'].append(accuracy_score(fold_trues, fold_preds))
    fold_metrics['Precision'].append(precision_score(fold_trues, fold_preds, average='macro', zero_division=0))
    fold_metrics['Recall'].append(recall_score(fold_trues, fold_preds, average='macro', zero_division=0))
    fold_metrics['F1_Score'].append(f1_score(fold_trues, fold_preds, average='macro', zero_division=0))
    
    fold_probs = np.array(fold_probs)
    fold_auc = roc_auc_score(fold_trues, fold_probs, multi_class='ovr', average='macro')
    fold_metrics['AUC'].append(fold_auc)

**Ensemble**

In [ ]:
print("\n" + "★"*50)
print("========== FINAL ENSEMBLE EVALUATION ==========")
print("★"*50)

ensemble_models = []
for path in fold_model_paths:
    m = DCAT_Model_True(num_classes=3).to(device)
    m.load_state_dict(torch.load(path))
    m.eval()
    ensemble_models.append(m)

all_preds, all_trues, all_probs = [], [], []

with torch.no_grad():
    for batch_a, batch_i, labels in test_loader:
        batch_a, batch_i = batch_a.to(device), batch_i.to(device)
        
        batch_probs = []
        for m in ensemble_models:
            outputs = m(batch_a, batch_i)
            probs = F.softmax(outputs, dim=1) 
            batch_probs.append(probs)
            
        avg_probs = torch.mean(torch.stack(batch_probs), dim=0)
        _, predicted = torch.max(avg_probs, 1)
        
        all_probs.extend(avg_probs.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())
        all_trues.extend(labels.numpy())

all_preds = np.array(all_preds)
all_trues = np.array(all_trues)
all_probs = np.array(all_probs)

print("Ensemble Performance on Fixed Test Set:")
print(classification_report(all_trues, all_preds, target_names=target_names))

ensemble_auc = roc_auc_score(all_trues, all_probs, multi_class='ovr', average='macro')
print(f"Grand Ensemble AUC (Macro OvR): {ensemble_auc:.4f}\n")

# BOOTSTRAP CONFIDENCE INTERVAL FOR ENSEMBLE
print("\n" + "★"*50)
print("========== ENSEMBLE BOOTSTRAP (1000 iterations) ==========")
print("★"*50)

n_bootstrap = 1000
np.random.seed(42)

boot_accuracy = []
boot_precision = []
boot_recall = []
boot_f1 = []
boot_auc = []

for i in range(n_bootstrap):
    indices = resample(np.arange(len(all_trues)), n_samples=len(all_trues), random_state=i)
    
    b_trues = all_trues[indices]
    b_preds = all_preds[indices]
    b_probs = all_probs[indices]
    
    # Skip if a bootstrap sample is missing a class (rare but possible)
    if len(np.unique(b_trues)) < 3:
        continue
    
    boot_accuracy.append(accuracy_score(b_trues, b_preds))
    boot_precision.append(precision_score(b_trues, b_preds, average='macro', zero_division=0))
    boot_recall.append(recall_score(b_trues, b_preds, average='macro', zero_division=0))
    boot_f1.append(f1_score(b_trues, b_preds, average='macro', zero_division=0))
    boot_auc.append(roc_auc_score(b_trues, b_probs, multi_class='ovr', average='macro'))

print("\n========== ENSEMBLE RESULTS WITH BOOTSTRAP ± STD ==========\n")

ensemble_boot_metrics = {
    'Accuracy':  (np.mean(boot_accuracy) * 100,  np.std(boot_accuracy) * 100),
    'Precision': (np.mean(boot_precision),        np.std(boot_precision)),
    'Recall':    (np.mean(boot_recall),           np.std(boot_recall)),
    'F1_Score':  (np.mean(boot_f1),               np.std(boot_f1)),
    'AUC':       (np.mean(boot_auc),             np.std(boot_auc)),
}

for metric, (mean_val, std_val) in ensemble_boot_metrics.items():
    if metric == 'Accuracy':
        print(f"  {metric:>12s}: {mean_val:.2f}% ± {std_val:.2f}%")
    elif metric == 'AUC':
        print(f"  {metric:>12s}: {mean_val:.4f} ± {std_val:.4f}")
    else:
        print(f"  {metric:>12s}: {mean_val:.4f} ± {std_val:.4f}")

# Also print 95% Confidence Intervals
print("\n========== ENSEMBLE 95% CONFIDENCE INTERVALS ==========\n")
for metric, values in zip(
    ['Accuracy', 'Precision', 'Recall', 'F1_Score', 'AUC'],
    [boot_accuracy, boot_precision, boot_recall, boot_f1, boot_auc]
):
    lower = np.percentile(values, 2.5)
    upper = np.percentile(values, 97.5)
    if metric == 'Accuracy':
        print(f"  {metric:>12s}: [{lower*100:.2f}%, {upper*100:.2f}%]")
    else:
        print(f"  {metric:>12s}: [{lower:.4f}, {upper:.4f}]")

# INDIVIDUAL FOLD METRICS (MEAN ± STD)
print("\n" + "★"*50)
print("========== INDIVIDUAL FOLD METRICS (MEAN ± STD) ==========")
print("★"*50)

metrics_summary = []
for metric in ['Accuracy', 'Precision', 'Recall', 'F1_Score', 'AUC']:
    mean_val = np.mean(fold_metrics[metric])
    std_val = np.std(fold_metrics[metric])
    metrics_summary.append({
        'Metric (Macro Avg)': metric,
        'Mean ± Std': f"{mean_val*100:.2f}% ± {std_val*100:.2f}%" if metric != 'AUC' else f"{mean_val:.4f} ± {std_val:.4f}"
    })

df_results = pd.DataFrame(metrics_summary)
print(df_results.to_string(index=False))

**Visualization & Plotting**

In [ ]:
# Plot Confusion Matrix
cm = confusion_matrix(all_trues, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names, annot_kws={"size": 14})
plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.ylabel('True Label', fontsize=12, fontweight='bold')
plt.title('Final Ensemble Confusion Matrix', fontsize=14, fontweight='bold')
plt.show()

# Plot Multi-Class ROC Curve
y_test_bin = label_binarize(all_trues, classes=[0, 1, 2])
n_classes = y_test_bin.shape[1]

fpr, tpr, roc_auc = dict(), dict(), dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], all_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(10, 8))
colors = cycle(['blue', 'red', 'green'])

for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f'ROC curve of class {target_names[i]} (area = {roc_auc[i]:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=12, fontweight='bold')
plt.title('Ensemble Multi-Class ROC Curve (One-vs-Rest)', fontsize=14, fontweight='bold')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()